In [ ]:
import sensors_util
import importlib    
import os
from natsort import natsorted    

In [ ]:
# Let's get all of the walkOutdoor recordings for 1 partiicpant first 
all_paths: list[str] = [] 

flic_raw_path: str = "/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026"

valid_subjects: int = 0
for idx, subject_dir in enumerate(natsorted(os.listdir(flic_raw_path))):
    if(valid_subjects >= 4):
        continue

    if(subject_dir == "FLIC_18"):
        continue

    if(subject_dir.startswith(".")):
        continue 

    subject_dir_path: str = os.path.join(flic_raw_path, subject_dir)

    if(not os.path.isdir(subject_dir_path)):
        continue
    
    for activity in os.listdir(subject_dir_path):
        if(activity.startswith(".")):
            continue 

        activity_path: str = os.path.join(subject_dir_path, activity, "GKA")
        assert os.path.exists(activity_path), f"Path: {activity_path} does not exist"

        all_paths.append(activity_path)
    
    valid_subjects += 1 
    

In [ ]:
importlib.reload(sensors_util)

frame_saturation_data_path = (
    sensors_util.LIGHT_LOGGER_ANALYSIS_ROOT
    / "testoutput"
    / "frame_saturation_calibration_data.npz"
)

df = sensors_util.fit_agc_to_illuminance(
    all_paths,
    illuminance_diagnostics=True,
    illuminance_diagnostics_output_dir=(
        sensors_util.LIGHT_LOGGER_ANALYSIS_ROOT
        / "testoutput"
        / "illuminance_channel_diagnostics"
    ),
    frame_saturation_data_output_path=frame_saturation_data_path,
)

In [ ]:
saturation_threshold_percent: float = 40.0
initial_samples_to_exclude: int = 0

frame_saturation_axes, activity_highlight_axes = (
    sensors_util.plot_frame_saturation_from_processed_data(
        processed_data_path=frame_saturation_data_path,
        maximum_saturation_percent=saturation_threshold_percent,
        minimum_correlation=0.9,
        initial_samples_to_exclude=initial_samples_to_exclude,
    )
)